In [1]:
import sys, torch
print(sys.executable)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

/home/zeus/miniconda3/envs/cloudspace/bin/python
True
NVIDIA L4


In [3]:
from unsloth import FastLanguageModel
import torch

base_model = "Qwen/Qwen3-4B-Thinking-2507"
lora_path = "models/lora_stage1_2"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = base_model,
    max_seq_length = 4096,
    dtype = None,
    load_in_4bit = True,
    trust_remote_code = True,
)

model.load_adapter(lora_path)

FastLanguageModel.for_inference(model)

print("Loaded base model + LoRA adapter")

Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2026.4.8: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/3.51G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

unsloth/qwen3-4b-thinking-2507-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Loading weights:   0%|          | 0/504 [00:00<?, ?it/s]

Loaded base model + LoRA adapter


# CSE 151B Competition — Starter Notebook

Welcome to the **CSE 151B Spring 2026 Math Reasoning Competition**!  
This notebook walks you through the full pipeline end-to-end:

1. Setting up the Python environment with `uv`
2. Loading the competition dataset
3. Running inference with **Qwen3-4B-Thinking** via vLLM (INT8 quantized)
4. Scoring responses against ground-truth answers
5. Saving results to JSONL for submission

The public dataset (`public.jsonl`) contains questions **with** answers so you can measure accuracy locally.  
The private test set used for the leaderboard does **not** include answers — for that, skip evaluation and submit the raw responses.

## 1. Environment Setup

We use [`uv`](https://github.com/astral-sh/uv) for fast, reproducible package management.

The steps below:
1. Install `uv` into `~/.local/bin`
2. Create a virtual environment at `.venv/`
3. Install all required packages (This might take a while)

> **After running this cell, restart the kernel** so that the newly installed packages (especially `vllm` and `transformers`) are picked up by the current Python session.

### Comment Out the cell below after first installation.

In [ ]:
# # Install uv
# !wget -qO- https://astral.sh/uv/install.sh | sh

# # Create a virtual environment
# !uv venv .venv --seed

# # Install dependencies — this is fast thanks to uv's parallel resolver
# !.venv/bin/python -m pip install sympy numpy transformers vllm tqdm bitsandbytes antlr4-python3-runtime==4.11.1 ipykernel jupyter

# # Install Jupyter Kernel
# !.venv/bin/python -m ipykernel install --user --name cse151b --display-name "Python (cse151b)"

# print("Done. Restart the kernel before proceeding.")
# print("Selection process: on top right, click on current kernel '(ususally named python)' -> 'select another kernel' -> 'Jupyter Kernel' -> 'Python (cse151b)'.")

### Run the cell below every time to activate the installed environment.

In [ ]:
# activate venv after installation. This needs to be run everytime.
!source ./.venv/bin/activate

In [ ]:
import sys
import os

# This points to the site-packages inside the virtual environment created by uv
venv_path = '/content/.venv/lib/python3.12/site-packages'

if os.path.exists(venv_path):
    # We insert it at index 0 so it takes priority over the system libraries
    sys.path.insert(0, venv_path)
    print("✅ sys.path updated! Using the TA's custom environment.")
else:
    print("❌ Error: .venv folder not found. Make sure the setup cell finished!")

# Quick check to see if vLLM is now visible
try:
    import vllm
    print(f"🚀 vLLM is ready! Version: {vllm.__version__}")
except ImportError:
    print("❌ vLLM still not found. Check if the venv_path is correct.")

✅ sys.path updated! Using the TA's custom environment.
🚀 vLLM is ready! Version: 0.19.0


## 2. Imports & Configuration

All key settings are collected in one place.  
- `DATA_PATH` — public dataset with ground-truth answers (use this to measure accuracy)
- `OUTPUT_PATH` — where per-question results will be written
- `GPU_ID` — which GPU to use (update if your machine has a different device index)
- `MAX_TOKENS` — maximum tokens the model may generate per response

In [ ]:
import json
import os
import re
import sys
from pathlib import Path
from typing import Optional

from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from tqdm import tqdm

# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507-FP8"
GPU_ID      = "0"                    # Fixed: Colab GPUs are indexed at 0
DATA_PATH = "/content/public.jsonl"
OUTPUT_PATH = "results/starter_results.jsonl"
MAX_TOKENS  = 16384

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

# Initialize the tokenizer here so it's available for build_prompt and apply_chat_template
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

/content/.venv/lib/python3.12/site-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

## 3. Load the Dataset

The dataset is stored as newline-delimited JSON (`.jsonl`). Each line is one question with the following fields:

| Field | Description |
|---|---|
| `id` | Unique question identifier |
| `question` | Problem statement |
| `options` | List of answer choices — present for **MCQ**, absent for **free-form** |
| `answer` | Ground-truth answer (letter for MCQ, value/list for free-form) |

In [ ]:
data = [json.loads(line) for line in open(DATA_PATH)]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options")   for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

# Preview one MCQ and one free-form item
mcq_sample  = next(d for d in data if d.get("options"))
free_sample = next(d for d in data if not d.get("options"))

print("\n── MCQ sample ──")
print(json.dumps(mcq_sample, indent=2))
print("\n── Free-form sample ──")
print(json.dumps(free_sample, indent=2))

Loaded 1126 questions  (375 MCQ, 751 free-form)

── MCQ sample ──
{
  "question": "$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $",
  "options": [
    "$0$",
    "$frac{1}{a}$",
    "$frac{3}{a}$",
    "$frac{1}{2a^2}$",
    "$frac{1}{2a}$",
    "$frac{2}{a}$",
    "$2a$",
    "$frac{3}{2a}$",
    "$frac{3}{2a^2}$",
    "$frac{1}{a^2}$"
  ],
  "answer": "F",
  "id": 1
}

── Free-form sample ──
{
  "question": "Find the sum of the first $325$ positive even whole numbers. Sum: [ANS]",
  "answer": [
    "325*(1+325)"
  ],
  "id": 0
}


## 4. Prompt Construction

We use two system prompts depending on the question type:

- **MCQ** — the model must select the best answer letter and wrap it in `\boxed{}`
- **Free-form** — the model solves step-by-step and puts the final answer in `\boxed{}`

`build_prompt()` returns the appropriate `(system, user)` pair for each item.

In [ ]:
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. Solve the problem step-by-step. "
    "Put your final answer inside \\boxed{}. "
    "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
    "e.g. \\boxed{3, 7}."
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician. "
    "Read the problem and the answer choices below, then select the single best answer. "
    "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
)


def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    """Return (system_prompt, user_prompt) for a question."""
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    return SYSTEM_PROMPT_MATH, question


# Verify with samples
for label, item in [("MCQ", mcq_sample), ("Free-form", free_sample)]:
    sys_p, usr_p = build_prompt(item["question"], item.get("options"))
    print(f"── {label} user prompt (first 200 chars) ──")
    print(usr_p[:200], "...\n")

── MCQ user prompt (first 200 chars) ──
$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $

Options:
A. $0$
B. $frac{1}{a}$
C. $frac{3}{a}$
D. $frac{1}{2a^2}$
E. $frac{1}{2a}$
F. $frac{2}{a}$
G. $2a$
H. $frac{3}{2a}$
I. $frac{3}{2a^2}$
J. ...

── Free-form user prompt (first 200 chars) ──
Find the sum of the first $325$ positive even whole numbers. Sum: [ANS] ...



In [ ]:
import os
import sys

# 1. Update the environment variable for all sub-processes
venv_site_packages = '/content/.venv/lib/python3.12/site-packages'
os.environ['PYTHONPATH'] = f"{venv_site_packages}:{os.environ.get('PYTHONPATH', '')}"

# 2. Make sure the venv bin is in the system PATH
os.environ['PATH'] = f"/content/.venv/bin:{os.environ['PATH']}"

# 3. Double check the sys.path (the hack you did earlier)
if venv_site_packages not in sys.path:
    sys.path.insert(0, venv_site_packages)

print("✅ Subprocess environment fixed. vLLM can now 'see' itself.")

✅ Subprocess environment fixed. vLLM can now 'see' itself.


## 5. Load Model with vLLM

We load **Qwen3-4B-Thinking-2507** with **INT8 quantization** via BitsAndBytes.  
Setting `load_format="bitsandbytes"` tells vLLM to apply on-the-fly INT8 weight quantization, roughly halving GPU memory usage compared to BF16.

Key parameters:
- `gpu_memory_utilization` — fraction of GPU VRAM reserved for the model and KV cache
- `max_model_len` — maximum sequence length (prompt + generation)
- `max_num_seqs` — maximum number of sequences processed in parallel

In [ ]:
from vllm import LLM, SamplingParams
import os

# Adjusting parameters to prevent EngineDeadError/OOM on L4 (24GB)
llm = LLM(
    model=MODEL_ID,
    quantization="fp8",
    kv_cache_dtype="fp8",
    gpu_memory_utilization=0.95, # Reduced to 0.90 for safety
    max_model_len=16384,
    trust_remote_code=True,
    enable_prefix_caching=False,
    max_num_seqs=256,
    max_num_batched_tokens=32768,
    enforce_eager=False
)

sampling_params = SamplingParams(
    max_tokens=12000,
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    min_p=0.0,
    presence_penalty=0.0,
    repetition_penalty=1.0,
)

print("vLLM re-initialized: Reduced concurrency to prevent OOM crashes.")


INFO 04-12 16:56:16 [utils.py:233] non-default args: {'trust_remote_code': True, 'kv_cache_dtype': 'fp8', 'max_model_len': 16384, 'enable_prefix_caching': False, 'gpu_memory_utilization': 0.95, 'max_num_batched_tokens': 32768, 'max_num_seqs': 256, 'disable_log_stats': True, 'quantization': 'fp8', 'model': 'Qwen/Qwen3-4B-Thinking-2507-FP8'}


config.json: 0.00B [00:00, ?B/s]

INFO 04-12 16:56:36 [model.py:549] Resolved architecture: Qwen3ForCausalLM
INFO 04-12 16:56:36 [model.py:1678] Using max model len 16384
INFO 04-12 16:56:37 [cache.py:227] Using fp8 data type to store kv cache. It reduces the GPU memory footprint and boosts the performance. Meanwhile, it may cause accuracy drop without a proper scaling factor.
INFO 04-12 16:56:37 [scheduler.py:238] Chunked prefill is enabled with max_num_batched_tokens=32768.
INFO 04-12 16:56:37 [vllm.py:790] Asynchronous scheduling is enabled.
INFO 04-12 16:56:37 [compilation.py:290] Enabled custom fusions: norm_quant, act_quant


generation_config.json:   0%|          | 0.00/240 [00:00<?, ?B/s]

WARNING 04-12 16:56:38 [system_utils.py:152] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
vLLM re-initialized: Reduced concurrency to prevent OOM crashes.


## 6. Generate Responses

We format every question into a chat-template prompt, then call `llm.generate()` in one batched pass.  
vLLM handles batching and scheduling internally — no manual batching needed.

In [ ]:

# Build prompts for first 5 entries
prompts = []
for item in data:
    system, user = build_prompt(item["question"], item.get("options"))
    prompt_text = tokenizer.apply_chat_template(
        [{"role": "system", "content": system},
         {"role": "user",   "content": user}],
        tokenize=False,
        add_generation_prompt=True,
    )
    prompts.append(prompt_text)

# Generate
print(f"Generating responses for {len(prompts)} questions...")
outputs = llm.generate(prompts, sampling_params=sampling_params)

responses = [out.outputs[0].text.strip() for out in outputs]

# Preview first 3
for i in range(min(3, len(responses))):
    print(f"\n── Response {i} (id={data[i].get('id')}) ──")
    print(responses[i][:400], "..." if len(responses[i]) > 400 else "")

Generating responses for 1126 questions...


Rendering prompts:   0%|          | 0/1126 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1126 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s…


── Response 0 (id=0) ──
This is a complex or challenging question, and it is difficult to provide a direct and correct answer. I need to think about it.
Well, so I need to find the sum of the first 325 positive even whole numbers. Hmm, let's start by recalling what the first few positive even whole numbers are to make sure I know what's being asked. Positive even whole numbers start at 2, right? So the first one is 2, se ...

── Response 1 (id=1) ──
Okay, let's try to figure out this integral. The problem is the integral from negative infinity to positive infinity of (a^(3/2)) divided by (s squared plus a squared) ds. Hmm, first, I need to recall how to integrate functions like 1/(s² + a²). I remember that the integral of 1/(s² + a²) ds is (1/a) arctan(s/a) + C. Let me confirm that. Yeah, the standard integral ∫1/(x² + c²) dx = (1/c) arctan(x ...

── Response 2 (id=2) ──
Okay, let's tackle this problem step by step. First, I remember that cooling problems like this are usually modeled

## 7. Score Responses

Scoring differs by question type:

- **MCQ**: extract the predicted letter from `\boxed{}` and compare to the gold letter (exact match).
- **Free-form**: use `Judger.auto_judge()` which handles symbolic and numeric equivalence.

Each result record contains `{id, is_mcq, gold, response, correct}`.

In [ ]:
def extract_letter(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""


def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()


# Load Judger for free-form scoring
import sys
sys.path.insert(0, "/content")
from judger import Judger
judger = Judger(strict_extract=False)

results = []
for item, response in tqdm(zip(data, responses), total=len(data), desc="Scoring"):
    is_mcq = bool(item.get("options"))
    gold   = item["answer"]

    if is_mcq:
        correct = score_mcq(response, str(gold))
    else:
        gold_list = gold if isinstance(gold, list) else [gold]
        try:
            correct = judger.auto_judge(
                pred=response,
                gold=gold_list,
                options=[[]] * len(gold_list),
            )
        except Exception:
            correct = False

    results.append({
        "id":       item.get("id"),
        "is_mcq":   is_mcq,
        "gold":     gold,
        "response": response,
        "correct":  correct,
    })

print(f"Scoring complete. {len(results)} results.")

Scoring: 100%|██████████| 1126/1126 [01:01<00:00, 18.17it/s]

Scoring complete. 1126 results.


## 8. Summary

Print accuracy broken down by question type.

In [ ]:
mcq_res  = [r for r in results if r["is_mcq"]]
free_res = [r for r in results if not r["is_mcq"]]

def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

print("=" * 50)
print("EVALUATION RESULTS")
print("=" * 50)
print(f"  MCQ        : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
print(f"  Free-form  : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
print(f"  Overall    : {sum(r['correct'] for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)")
print("=" * 50)

EVALUATION RESULTS
  MCQ        :  244 /  375  (65.07%)
  Free-form  :  407 /  751  (54.19%)
  Overall    :  651 / 1126  (57.82%)


In [ ]:
import random
import json
from pathlib import Path

# Path to the results file you provided
RESULTS_FILE = '/content/starter_results_12k.jsonl'
DATA_PATH = '/content/public.jsonl'  # Original data for questions/options

# 1. Load results from the saved file
if Path(RESULTS_FILE).exists():
    results = [json.loads(line) for line in open(RESULTS_FILE)]
    print(f"✅ Loaded {len(results)} results from {RESULTS_FILE}")
else:
    print(f"❌ Error: {RESULTS_FILE} not found. Please ensure the file is uploaded.")
    results = []

# 2. Load original dataset for question context if available
data_map = {}
if Path(DATA_PATH).exists():
    data_map = {item['id']: item for item in [json.loads(line) for line in open(DATA_PATH)]}
else:
    print(f"⚠️ Warning: {DATA_PATH} not found. Question text might be missing.")

# 3. Analyze errors
if results:
    incorrect_results = [r for r in results if not r.get('correct', False)]
    print(f"Total incorrect: {len(incorrect_results)}")
    print("Showing 5 random errors:\n" + "="*50)

    sample_errors = random.sample(incorrect_results, min(5, len(incorrect_results)))

    for i, res in enumerate(sample_errors):
        orig_item = data_map.get(res['id'], {})
        print(f"Error #{i+1} | ID: {res['id']} | MCQ: {res.get('is_mcq')}")
        print(f"QUESTION: {orig_item.get('question', 'Question text not found in public.jsonl')}")

        if res.get('is_mcq'):
            options = orig_item.get('options', [])
            if options:
                labels = [chr(65 + j) for j in range(len(options))]
                print("OPTIONS:")
                for lbl, opt in zip(labels, options):
                    print(f"  {lbl}. {opt}")

        print(f"GOLD ANSWER: {res.get('gold')}")
        print(f"MODEL RESPONSE :\n{res.get('response', '')}...")
        print("-"*30)


✅ Loaded 1126 results from /content/starter_results_12k.jsonl
Total incorrect: 475
Showing 5 random errors:
Error #1 | ID: 593 | MCQ: True
QUESTION: Given that 695xy155 is a multiple of 99, find $x, \mathbf{y}$.
OPTIONS:
  A. x=0,y=8
  B. x=4,y=7
  C. x=1,y=4
  D. x=8,y=0
  E. x=6,y=9
  F. x=9,y=6
  G. x=3,y=2
  H. x=2,y=3
  I. x=5,y=1
  J. x=7,y=5
GOLD ANSWER: H
MODEL RESPONSE :
Okay, let's see. The problem says that 695xy155 is a multiple of 99. I need to find x and y. Hmm, 99 is 9 times 11, right? So if a number is divisible by 99, it must be divisible by both 9 and 11. That makes sense because 9 and 11 are coprime. So I should check the divisibility rules for 9 and 11.

First, let's recall the divisibility rule for 9: the sum of the digits must be a multiple of 9. And for 11: the difference between the sum of the digits in the odd positions and the sum of the digits in the even positions must be a multiple of 11 (including zero).

Let me write down the number: 6 9 5 x y 1 5 5. Wait

In [ ]:
ERROR_LOG_PATH = 'detailed_errors.txt'

with open(ERROR_LOG_PATH, 'w') as f:
    f.write(f"--- ERROR ANALYSIS REPORT ({len(incorrect_results)} failures) ---\n\n")

    for i, res in enumerate(incorrect_results):
        orig_item = data_map.get(res['id'], {})
        f.write(f"{'='*80}\n")
        f.write(f"FAILURE #{i+1} | ID: {res['id']} | MCQ: {res.get('is_mcq')}\n")
        f.write(f"QUESTION: {orig_item.get('question', 'Question text not found in public.jsonl')}\n\n")

        if res.get('is_mcq'):
            options = orig_item.get('options', [])
            if options:
                labels = [chr(65 + j) for j in range(len(options))]
                f.write("OPTIONS:\n")
                for lbl, opt in zip(labels, options):
                    f.write(f"  {lbl}. {opt}\n")
                f.write("\n")

        f.write(f"GOLD ANSWER: {res.get('gold')}\n")
        f.write(f"MODEL RESPONSE :\n{res.get('response', '')}\n")
        f.write(f"{'='*80}\n\n")

print(f"✅ Detailed error log saved to {ERROR_LOG_PATH}")

✅ Detailed error log saved to detailed_errors.txt


In [ ]:
# import json
# import random
# import re
# import sys
# from pathlib import Path

# # Reload original dataset to get questions, options, and gold answers
# DATA_PATH = "/content/public.jsonl"
# data_map = {}
# if Path(DATA_PATH).exists():
#     with open(DATA_PATH, 'r') as f:
#         for line in f:
#             item = json.loads(line.strip())
#             data_map[item['id']] = item

# # Target your newly uploaded file
# BASELINE_FILE = '/content/baseline_answers.txt'

# # --- Scoring Functions ---
# def extract_letter(text: str) -> str:
#     m = re.search(r"\\boxed\{([A-Za-z])\}", text)
#     if m:
#         return m.group(1).upper()
#     matches = re.findall(r"\b([A-Z])\b", text.upper())
#     return matches[-1] if matches else ""

# def score_mcq(response: str, gold_letter: str) -> bool:
#     return extract_letter(response) == gold_letter.strip().upper()

# try:
#     if "/content" not in sys.path:
#         sys.path.insert(0, "/content")
#     from judger import Judger
#     judger = Judger(strict_extract=False)
# except Exception as e:
#     print(f"Warning: Could not load Judger: {e}")
#     judger = None

# if not Path(BASELINE_FILE).exists():
#     print(f"❌ Error: {BASELINE_FILE} not found. Please ensure you uploaded it.")
# else:
#     results = []
#     with open(BASELINE_FILE, 'r') as f:
#         for line in f:
#             line = line.strip()
#             if line.startswith('{'):
#                 try:
#                     results.append(json.loads(line))
#                 except json.JSONDecodeError:
#                     continue

#     if not results:
#         print("❌ Error: No valid JSON records found in the file.")
#     else:
#         mcq_errors = []
#         free_form_errors = []

#         for r in results:
#             q_id = r.get('id')
#             if q_id not in data_map:
#                 continue

#             orig_item = data_map[q_id]
#             is_mcq = bool(orig_item.get("options"))
#             gold = orig_item["answer"]
#             response = r.get("response", "")

#             # Evaluate on the fly
#             correct = False
#             if is_mcq:
#                 correct = score_mcq(response, str(gold))
#             elif judger:
#                 gold_list = gold if isinstance(gold, list) else [gold]
#                 try:
#                     correct = judger.auto_judge(pred=response, gold=gold_list, options=[[]] * len(gold_list))
#                 except Exception:
#                     correct = False

#             if not correct:
#                 # Attach original data for display
#                 r['question'] = orig_item.get('question')
#                 r['options'] = orig_item.get('options', [])
#                 r['gold'] = gold

#                 if is_mcq:
#                     mcq_errors.append(r)
#                 else:
#                     free_form_errors.append(r)

#         print(f"Found {len(mcq_errors)} MCQ errors and {len(free_form_errors)} free-form errors in the file.")

#         # Sample 6 from each if available
#         sample_mcq = random.sample(mcq_errors, min(6, len(mcq_errors)))
#         sample_free = random.sample(free_form_errors, min(6, len(free_form_errors)))

#         def print_detailed_error(res_list, title):
#             print(f"\n{'='*30} {title} {'='*30}")
#             for i, res in enumerate(res_list):
#                 print(f"\n[FAILURE #{i+1}] ID: {res.get('id')}")
#                 print(f"QUESTION:\n{res.get('question')}")

#                 options = res.get('options', [])
#                 if options:
#                     labels = [chr(65 + j) for j in range(len(options))]
#                     print("OPTIONS:")
#                     for lbl, opt in zip(labels, options):
#                         print(f"  {lbl}. {opt}")

#                 print(f"\nGOLD ANSWER: {res.get('gold')}")
#                 print(f"\nMODEL FULL RESPONSE:\n{'-'*20}\n{res.get('response')}\n{'-'*20}")
#                 print(f"{'*'*80}")

#         # Execute analysis
#         print_detailed_error(sample_mcq, "MCQ ERRORS")
#         print_detailed_error(sample_free, "OPEN-ENDED ERRORS")

## 9. Save Results

Results are written as newline-delimited JSON.

**With evaluation** (public set — you have ground-truth):  
Each line: `{id, is_mcq, gold, response, correct}`

**Without evaluation** (private test set — no ground-truth available):  
Each line: `{id, is_mcq, response}` — omit `gold` and `correct`.

Toggle `SAVE_EVAL` below accordingly.

In [ ]:
SAVE_EVAL = True   # Set to False when running on the private test set

out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w") as f:
    for r in results:
        if SAVE_EVAL:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "gold": r["gold"],
                      "response": r["response"], "correct": r["correct"]}
        else:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "response": r["response"]}
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(results)} records to {out_path}")

Saved 1126 records to results/starter_results.jsonl


## Next Steps

This notebook gives you a working baseline. Here are directions to improve your score:

- **Prompt engineering** — try different system prompts or few-shot examples inside the user turn
- **Sampling parameters** — adjust `temperature`, `top_p`, or use majority voting across multiple samples
- **Fine-tuning** — the competition allows model fine-tuning; see the course resources for guidance

Good luck!

**Reasoning**:
I will add a new code cell at the end of the notebook to install the required libraries dspy-ai and openai as per the subtask instructions.



## SFT Environment Creation

### Subtask:
Create a dedicated virtual environment to isolate SFT libraries from the inference environment.


**Reasoning**:
I will create a new virtual environment using uv, verify its creation, and update the system environment variables (PYTHONPATH and PATH) to prioritize this new environment for the SFT task.



In [8]:
import os
import sys

# 1. Create a dedicated virtual environment for SFT
print("Creating .sft_venv...")
!uv venv .sft_venv --seed --clear

# 2. Verify creation
sft_venv_path = os.path.abspath('.sft_venv')
if os.path.exists(sft_venv_path):
    print(f"✅ Virtual environment created at: {sft_venv_path}")
else:
    print("❌ Error: .sft_venv creation failed.")

# 3. Update PYTHONPATH to include the SFT venv site-packages
# Note: Assuming python 3.12 as used in previous steps
sft_site_packages = os.path.join(sft_venv_path, 'lib', 'python3.12', 'site-packages')

os.environ['PYTHONPATH'] = f"{sft_site_packages}:{os.environ.get('PYTHONPATH', '')}"
if sft_site_packages not in sys.path:
    sys.path.insert(0, sft_site_packages)

# 4. Update PATH to prioritize .sft_venv/bin
sft_bin = os.path.join(sft_venv_path, 'bin')
os.environ['PATH'] = f"{sft_bin}:{os.environ.get('PATH', '')}"

print("✅ Environment variables updated. PYTHONPATH and PATH now prioritize .sft_venv.")

Creating .sft_venv...
Using CPython 3.12.13 interpreter at: /usr/bin/python3
Creating virtual environment with seed packages at: .sft_venv
 + pip==26.0.1
Activate with: source .sft_venv/bin/activate
✅ Virtual environment created at: /content/.sft_venv
✅ Environment variables updated. PYTHONPATH and PATH now prioritize .sft_venv.


In [14]:
import os
import sys
import importlib

# 1. Force prioritize the SFT venv site-packages
sft_site_packages = '/content/.sft_venv/lib/python3.12/site-packages'
if sft_site_packages in sys.path:
    sys.path.remove(sft_site_packages)
sys.path.insert(0, sft_site_packages)

# 2. Install compatible versions and fix torchvision CUDA mismatch
print("Installing libraries in .sft_venv...")
!/content/.sft_venv/bin/pip install -q --force-reinstall \
    transformers==4.44.0 \
    accelerate==0.33.0 \
    trl==0.10.1 \
    bitsandbytes \
    peft \
    datasets \
    torchvision --extra-index-url https://download.pytorch.org/whl/cu124

# 3. Deep clean sys.modules to prevent cross-contamination from the previous environment
modules_to_remove = [k for k in sys.modules if any(k.startswith(prefix) for prefix in ['transformers', 'trl', 'accelerate', 'peft', 'bitsandbytes', 'tokenizers', 'huggingface_hub', 'torchvision'])]
for mod in modules_to_remove:
    del sys.modules[mod]

# Force Python to re-scan sys.path to find the newly installed modules
importlib.invalidate_caches()

try:
    import transformers
    import accelerate
    from trl import SFTTrainer
    import bitsandbytes as bnb

    print(f"\u2705 bitsandbytes version: {bnb.__version__}")
    print(f"\u2705 transformers version: {transformers.__version__}")
    print(f"\u2705 SFTTrainer imported successfully!")
    print("\U0001f680 SFT environment is now isolated and ready!")
except Exception as e:
    print(f"\u274c Verification failed: {e}")
    print("Checking for specific missing components...")
    !/content/.sft_venv/bin/pip list | grep -E 'transformers|trl|accelerate'


Installing libraries in .sft_venv...
❌ Verification failed: Failed to import trl.trainer.sft_trainer because of the following error (look up to see its traceback):
Failed to import transformers.trainer because of the following error (look up to see its traceback):
cannot import name 'clear_device_cache' from 'accelerate.utils.memory' (/content/.sft_venv/lib/python3.12/site-packages/accelerate/utils/memory.py)
Checking for specific missing components...
accelerate             0.30.1
transformers           4.40.1
trl                    0.8.6


In [15]:
import os
import sys
import importlib

try:
    sft_site_packages = '/content/.sft_venv/lib/python3.12/site-packages'
    if sft_site_packages not in sys.path:
        sys.path.insert(0, sft_site_packages)
    import transformers
    import accelerate
    from trl import SFTTrainer
    import bitsandbytes as bnb

    print(f"✅ bitsandbytes version: {bnb.__version__}")
    print(f"✅ transformers version: {transformers.__version__}")
    print(f"✅ SFTTrainer imported successfully!")
    print("🚀 SFT environment is now isolated and ready!")
except Exception as e:
    print(f"❌ Verification failed: {e}")
    print("Checking for specific missing components...")
    !/content/.sft_venv/bin/pip list | grep -E 'transformers|trl|accelerate'

❌ Verification failed: Failed to import trl.trainer.sft_trainer because of the following error (look up to see its traceback):
Failed to import transformers.trainer because of the following error (look up to see its traceback):
cannot import name 'clear_device_cache' from 'accelerate.utils.memory' (/content/.sft_venv/lib/python3.12/site-packages/accelerate/utils/memory.py)
Checking for specific missing components...
accelerate             0.30.1
transformers           4.40.1
trl                    0.8.6


In [ ]:
# # Force reinstall of stable CUDA 13.0 compatible versions
# !/content/.sft_venv/bin/pip install --force-reinstall \
#     torch==2.10.0 \
#     torchvision==0.25.0 \
#     torchaudio==2.10.0 \
#     --index-url https://download.pytorch.org/whl/cu130


SFT Starts here

In [ ]:
# mount google drive
from google.colab import drive
drive.mount('/content/drive')

import os
import json
os.makedirs('/content/drive/MyDrive/checkpoints', exist_ok=True)

results_log = {}


Mounted at /content/drive


In [ ]:

requirements_file = '/content/requirements_sft_final.txt'
sft_pip = '/content/.sft_venv/bin/pip'

if os.path.exists(requirements_file):
    print(f"Installing dependencies from {requirements_file}...")
    !{sft_pip} install -r {requirements_file}
    print("✅ Dependencies installed successfully.")
else:
    print(f"⚠️ Warning: {requirements_file} not found. Please upload it if you want to install specific pinned versions.")

In [ ]:
!/content/.sft_venv/bin/pip install --no-deps "unsloth[cu130-torch2100] @ git+https://github.com/unslothai/unsloth.git"

In [ ]:
import sys
sys.path.insert(0, '/content/.sft_venv/lib/python3.12/site-packages')


import torch
import torchvision
print(f"Torch CUDA: {torch.version.cuda}")
print(f"TorchVision: {torchvision.__version__}")
print(f"GPU Available: {torch.cuda.is_available()}")

import transformers
import accelerate
from trl import SFTTrainer
import bitsandbytes as bnb

print(f"✅ bitsandbytes version: {bnb.__version__}")
print(f"✅ transformers version: {transformers.__version__}")
print(f"✅ SFTTrainer imported successfully!")



Torch CUDA: 13.0
TorchVision: 0.25.0+cu130
GPU Available: True
✅ bitsandbytes version: 0.49.2
✅ transformers version: 5.5.4
✅ SFTTrainer imported successfully!


Downloading the datasets

In [ ]:
import re
from datasets import load_dataset, concatenate_datasets, Dataset

print("Loading additional math datasets...")

# 1. Load the requested datasets
dataset_math = load_dataset("HuggingFaceH4/MATH", split="train")
dataset_s1k = load_dataset("simplescaling/s1K-1.1", split="train")
dataset_numina = load_dataset("AI-MO/NuminaMath-CoT", split="train")
dataset_stratos = load_dataset("bespokelabs/Bespoke-Stratos-17k", split="train")
dataset_openr1 = load_dataset("open-r1/OpenR1-Math-220k", "default", split="train")

# 2. Normalization Function
def extract_boxed_answer(text):
    if not text: return ""
    # Simple extraction for \boxed{}
    matches = re.findall(r'\\boxed\{((?:[^{}]|(?:\{[^{}]*\}))*)\}', text)
    if not matches:
        matches = re.findall(r'\\boxed\{(.*?)\}', text)
    return matches[-1] if matches else ""

def normalize_dataset(example, ds_name):
    res = {"problem": "", "solution": "", "answer": "", "level": "unknown", "source": ds_name}

    if ds_name == "math":
        res["problem"] = example.get("problem", "")
        res["solution"] = example.get("solution", "")
        res["level"] = str(example.get("level", "unknown"))
        res["answer"] = extract_boxed_answer(res["solution"])

    elif ds_name == "s1k":
        res["problem"] = example.get("question", "")
        res["solution"] = (example.get("deepseek_thinking_trajectory") or
                          example.get("gemini_thinking_trajectory") or "")
        attempt = (example.get("deepseek_attempt") or
                  example.get("gemini_attempt") or "")
        res["answer"] = extract_boxed_answer(attempt)

    elif ds_name == "numina":
        messages = example.get("messages", [])
        user_msg = next((m["content"] for m in messages if m["role"] == "user"), "")
        assistant_msg = next((m["content"] for m in messages if m["role"] == "assistant"), "")
        res["problem"] = example.get("problem", "") or user_msg
        res["solution"] = example.get("solution", "") or assistant_msg
        res["answer"] = extract_boxed_answer(res["solution"])
        res["level"] = example.get("source", "unknown")

    elif ds_name == "stratos":
        conversations = example.get("conversations", [])
        if len(conversations) >= 2:
            res["problem"] = conversations[0].get("value", conversations[0].get("content", ""))
            res["solution"] = conversations[1].get("value", conversations[1].get("content", ""))
        res["answer"] = extract_boxed_answer(res["solution"])

    elif ds_name == "openr1":
        res["problem"] = example.get("problem", "")
        # Handle the list type in generations
        generations = example.get("generations", [])
        if isinstance(generations, list) and len(generations) > 0:
            res["solution"] = generations[0]
        else:
            res["solution"] = str(generations)
        res["answer"] = example.get("answer", "")
        res["level"] = example.get("source", "unknown") #this gives stuff like olympiads, cn_k1 etc.

    return res

# Map and Clean
print("Normalizing fields...")
ds_math_clean = dataset_math.map(lambda x: normalize_dataset(x, "math"), remove_columns=dataset_math.column_names)
ds_s1k_clean = dataset_s1k.map(lambda x: normalize_dataset(x, "s1k"), remove_columns=dataset_s1k.column_names)
ds_numina_clean = dataset_numina.map(lambda x: normalize_dataset(x, "numina"), remove_columns=dataset_numina.column_names)
ds_stratos_clean = dataset_stratos.map(lambda x: normalize_dataset(x, "stratos"), remove_columns=dataset_stratos.column_names)
ds_openr1_clean = dataset_openr1.map(lambda x: normalize_dataset(x, "openr1"), remove_columns=dataset_openr1.column_names)

# 3. Concatenate into a unified pool
all_math_data = concatenate_datasets([ds_math_clean, ds_s1k_clean, ds_numina_clean, ds_stratos_clean, ds_openr1_clean])

print(f"\u2705 All datasets loaded and unified.")
print(f"Total unified samples: {len(all_math_data)}")

Loading additional math datasets...


Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Normalizing fields...
✅ All datasets loaded and unified.
Total unified samples: 971683


In [ ]:
# Check each dataset before merging
DATASETS = [
            # ("math", ds_math_clean),
            # ("s1k", ds_s1k_clean),
            # ("numina", ds_numina_clean),
            # ("stratos", ds_stratos_clean),
            ("openr1", ds_openr1_clean)
            ]
for name, ds in DATASETS:
    total = len(ds)
    if name == "s1k":
        min_w, max_w = 500, 8000    # thinking traces, intentionally long
    elif name == "openr1":
        min_w, max_w = 100, 5000    # DeepSeek-R1 traces, moderate-long
    elif name == "numina":
        min_w, max_w = 30, 1500     # plain CoT, shorter
    elif name == "stratos":
        min_w, max_w = 150, 3500    # complex reasoning
    elif name == "math":
        min_w, max_w = 50, 2500     # MATH dataset solutions, short-medium
    else:
        min_w, max_w = 100, 2000    # fallback
    has_problem = sum(1 for x in ds if len(x["problem"]) > 10)
    has_boxed = sum(1 for x in ds if bool(x["answer"]))
    in_word_range = sum(1 for x in ds if min_w < len(x["solution"].split()) < max_w)

    print(f"\n--- {name} ({total} samples) ---")
    print(f"  has_problem:    {has_problem} ({100*has_problem/total:.1f}%)")
    print(f"  has_boxed:      {has_boxed} ({100*has_boxed/total:.1f}%)")
    print(f"  word_range:     {in_word_range} ({100*in_word_range/total:.1f}%)")
    print(f"  sample problem: {ds[0]['problem'][:80]}")
    print(f"  sample answer:  {ds[0]['answer']}")
    print(f"  sample words:   {ds[0]['solution'][:100]}")


--- openr1 (93733 samples) ---
  has_problem:    93733 (100.0%)
  has_boxed:      93733 (100.0%)
  word_range:     76436 (81.5%)
  sample problem: ## Task B-1.3.

A ship traveling along a river has covered $24 \mathrm{~km}$ ups
  sample answer:  v_{R}=4\mathrm{~}/\mathrm{},v_{B}=10\mathrm{~}/\mathrm{}
  sample words:   <think>
Okay, so I need to find the speed of the ship in still water and the speed of the river. Let


In [ ]:
# Check raw numina source values
from collections import Counter
# print(Counter(all_math_data["source"]))
# print(len(ds_openr1_clean))
# print(ds_openr1_clean[0]["source"])
print(ds_openr1_clean[0]["source"])  # should print "openr1"
print(ds_openr1_clean[0]["level"])   # should print "olympiads" or similar
print(ds_numina_clean[0]["level"])   # should print "cn_k12" or similar

openr1
olympiads
synthetic_math


In [ ]:
import collections
import re
from datasets import concatenate_datasets
import random

# Initialize a nested counter to track distribution
stage_distribution = collections.defaultdict(lambda: collections.defaultdict(int))

def is_valid(example):
    words = len(example["solution"].split())
    has_boxed = bool(example["answer"])
    has_problem = len(example["problem"]) > 10
    max_words = 5000 if example["source"] == "s1k" else 2000
    return 100 < words < max_words and has_boxed and has_problem

# 1. Filter validity across unified data
all_math_data = all_math_data.filter(is_valid)
print(f"After validity filter: {len(all_math_data)}")

def get_math_level(example):
    lvl = example["level"]
    match = re.search(r'\d+', str(lvl))
    return int(match.group()) if match else 0

def assign_stage(example):
    src = example["source"]
    stage = 2 # default fallback
    words = len(example["solution"].split())

    if src == "math":
        lvl = get_math_level(example)
        if lvl <= 2: stage = 1
        elif lvl == 3: stage = 2
        elif lvl == 4: stage = 3
        else: stage = 4

    elif src == "s1k":
        stage = 4

    elif src == "numina":
        # Use the sub-source field stored in 'level' during normalization for Numina
        sub = str(example.get("level", "")).lower()

        # Stage 1 — easy, high solve rate / basic K12
        if any(k in sub for k in ["cn_k12", "synthetic_math", "orca_math", "gsm8k"]):
            stage = 1
        # Stage 2 — medium competition / standard math
        elif any(k in sub for k in ["synthetic_amc", "math"]):
            stage = 2
        # Stage 3 — hard competition / forum level
        elif any(k in sub for k in ["aops_forum"]):
            stage = 3
        # Stage 4 — olympiad / high complexity
        elif any(k in sub for k in ["olympiads", "amc_aime"]):
            stage = 4
        else:
            # Fallback to word count for unknown Numina sub-sources
            if words < 150: stage = 1
            elif words < 400: stage = 2
            elif words < 800: stage = 3
            else: stage = 4

    elif src == "stratos":
        stage = 5

    elif src == "openr1":
        # open-r1 usually contains high quality reasoning, placing in Stage 3/4
        stage = 4 if words > 1000 else 3

    return {"stage": stage}

# 2. Map stages to all data
all_math_data = all_math_data.map(assign_stage)

# 3. Subsample Numina to prevent dominance
NUMINA_CAPS = {1: 20000, 2: 20000, 3: 10000, 4: 10000}

def subsample_by_source_and_stage(ds, source_name, caps):
    final_subsets = []
    # Keep all non-target source data
    others = ds.filter(lambda x: x["source"] != source_name)
    final_subsets.append(others)

    # Subsample target source (Numina)
    target_ds = ds.filter(lambda x: x["source"] == source_name)
    for s, cap in caps.items():
        subset = target_ds.filter(lambda x: x["stage"] == s)
        n = min(len(subset), cap)
        if n > 0:
            sampled = subset.shuffle(seed=42).select(range(n))
            final_subsets.append(sampled)
            print(f"{source_name} Stage {s}: Subsampled to {n}")

    return concatenate_datasets(final_subsets)

all_math_data = subsample_by_source_and_stage(all_math_data, "numina", NUMINA_CAPS)

# 4. Final Stage Distribution Reporting
stage_distribution = collections.defaultdict(lambda: collections.defaultdict(int))
for x in all_math_data:
    stage_distribution[x["source"]][x["stage"]] += 1

print("\n--- Final Curriculum Stage Distribution ---")
header = f"{'Source':<15} | {'Stage 1':<8} | {'Stage 2':<8} | {'Stage 3':<8} | {'Stage 4':<8} | {'Stage 5':<8}"
print(header)
print("-" * len(header))
for src, counts in sorted(stage_distribution.items()):
    print(f"{src:<15} | {counts[1]:<8} | {counts[2]:<8} | {counts[3]:<8} | {counts[4]:<8} | {counts[5]:<8}")

# 5. Final Splits
stage_1 = all_math_data.filter(lambda x: x["stage"] == 1)
stage_2 = all_math_data.filter(lambda x: x["stage"] == 2)
stage_3 = all_math_data.filter(lambda x: x["stage"] == 3)
stage_4 = all_math_data.filter(lambda x: x["stage"] == 4)
stage_5_replay = all_math_data.filter(lambda x: x["stage"] == 5)

def sample_fraction(ds, frac=0.1, seed=42):
    if len(ds) == 0: return ds
    n = max(1, int(len(ds) * frac))
    idx = random.Random(seed).sample(range(len(ds)), min(n, len(ds)))
    return ds.select(idx)

stage_5_replay = concatenate_datasets([
    stage_5_replay,
    sample_fraction(stage_1),
    sample_fraction(stage_2),
    sample_fraction(stage_3),
    sample_fraction(stage_4),
])

print(f"\nFinal Stage Splits:\n- Stage 1: {len(stage_1)}\n- Stage 2: {len(stage_2)}\n- Stage 3: {len(stage_3)}\n- Stage 4: {len(stage_4)}\n- Stage 5 (Replay): {len(stage_5_replay)}")

After validity filter: 700604
numina Stage 1: Subsampled to 20000
numina Stage 2: Subsampled to 20000
numina Stage 3: Subsampled to 10000
numina Stage 4: Subsampled to 10000

--- Final Curriculum Stage Distribution ---
Source          | Stage 1  | Stage 2  | Stage 3  | Stage 4  | Stage 5 
----------------------------------------------------------------------
math            | 39       | 79       | 126      | 157      | 0       
numina          | 20000    | 20000    | 10000    | 10000    | 0       
openr1          | 0        | 0        | 9308     | 26438    | 0       
s1k             | 0        | 0        | 0        | 522      | 0       
stratos         | 0        | 0        | 0        | 0        | 8063    


Filter:   0%|          | 0/104732 [00:00<?, ? examples/s]

Filter:   0%|          | 0/104732 [00:00<?, ? examples/s]

Filter:   0%|          | 0/104732 [00:00<?, ? examples/s]

Filter:   0%|          | 0/104732 [00:00<?, ? examples/s]

Filter:   0%|          | 0/104732 [00:00<?, ? examples/s]


Final Stage Splits:
- Stage 1: 20039
- Stage 2: 20079
- Stage 3: 19434
- Stage 4: 37117
- Stage 5 (Replay): 17727


In [ ]:
# save dataset to google drive
import os

base_dir = '/content/drive/MyDrive/math_sft_datasets'
print(f"Saving datasets to {base_dir}...")

# Save each stage dataset to disk
stage_1.save_to_disk(os.path.join(base_dir, 'stage_1'))
print("✅ stage_1 saved")

stage_2.save_to_disk(os.path.join(base_dir, 'stage_2'))
print("✅ stage_2 saved")

stage_3.save_to_disk(os.path.join(base_dir, 'stage_3'))
print("✅ stage_3 saved")

stage_4.save_to_disk(os.path.join(base_dir, 'stage_4'))
print("✅ stage_4 saved")

stage_5_replay.save_to_disk(os.path.join(base_dir, 'stage_5_replay'))
print("✅ stage_5_replay saved")

print("All datasets successfully saved to Google Drive!")

Saving datasets to /content/drive/MyDrive/math_sft_datasets...


Saving the dataset (0/1 shards):   0%|          | 0/20039 [00:00<?, ? examples/s]

✅ stage_1 saved


Saving the dataset (0/1 shards):   0%|          | 0/20079 [00:00<?, ? examples/s]

✅ stage_2 saved


...

✅ stage_3 saved


Saving the dataset (0/1 shards):   0%|          | 0/37117 [00:00<?, ? examples/s]

✅ stage_4 saved


Saving the dataset (0/1 shards):   0%|          | 0/17727 [00:00<?, ? examples/s]

✅ stage_5_replay saved
All datasets successfully saved to Google Drive!


In [ ]:
# build the validation dataset :
from datasets import load_dataset
import random

# Use MATH test split — completely separate from train
math_test = load_dataset("HuggingFaceH4/MATH", split="test")

# Stratify by level
easy_problems   = [x for x in math_test if x["level"] in ["Level 1", "Level 2"]]
medium_problems = [x for x in math_test if x["level"] in ["Level 3", "Level 4"]]
hard_problems   = [x for x in math_test if x["level"] == "Level 5"]

# Sample fixed eval set — same every run
rng = random.Random(42)
eval_easy   = rng.sample(easy_problems,   min(50, len(easy_problems)))
eval_medium = rng.sample(medium_problems, min(50, len(medium_problems)))
eval_hard   = rng.sample(hard_problems,   min(50, len(hard_problems)))

eval_set = {
    "easy":   eval_easy,
    "medium": eval_medium,
    "hard":   eval_hard,
}

print(f"Eval set: {len(eval_easy)} easy, {len(eval_medium)} medium, {len(eval_hard)} hard")

# Save to drive — load this every session, never regenerate
import json
with open(f"{base_dir}/eval_set.json", "w") as f:
    json.dump(eval_set, f)

eval_set.save_to_disk(os.path.join(base_dir, 'eval_set'))

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/351k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/240k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/746 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/546 [00:00<?, ? examples/s]

Eval set: 50 easy, 50 medium, 50 hard


AttributeError: 'dict' object has no attribute 'save_to_disk'

In [ ]:
import os
import json
from datasets import Dataset, DatasetDict

# Define the path on Google Drive
eval_save_path = os.path.join(base_dir, 'eval_set')

# 1. Save as a Hugging Face DatasetDict for high-performance loading
# We convert our dictionary of lists into a DatasetDict
eval_dataset_dict = DatasetDict({
    k: Dataset.from_list(v) for k, v in eval_set.items()
})

eval_dataset_dict.save_to_disk(eval_save_path)
print(f"✅ eval_set saved as Dataset to: {eval_save_path}")

# 2. Save as a JSON file for manual inspection
json_save_path = os.path.join(base_dir, 'eval_set.json')
with open(json_save_path, 'w') as f:
    json.dump(eval_set, f, indent=2)

print(f"✅ eval_set saved as JSON to: {json_save_path}")

Saving the dataset (0/1 shards):   0%|          | 0/50 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/50 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/50 [00:00<?, ? examples/s]

✅ eval_set saved as Dataset to: /content/drive/MyDrive/math_sft_datasets/eval_set
✅ eval_set saved as JSON to: /content/drive/MyDrive/math_sft_datasets/eval_set.json


Run the cell below to load the already downloaded datasets and also download dependencies

In [ ]:
# load dataset from google drive
from datasets import load_from_disk
import os
from google.colab import drive
drive.mount('/content/drive')



base_dir = '/content/drive/MyDrive/math_sft_datasets'
print(f"Loading curriculum stages from {base_dir}...")

try:
    stage_1 = load_from_disk(os.path.join(base_dir, 'stage_1'))
    stage_2 = load_from_disk(os.path.join(base_dir, 'stage_2'))
    stage_3 = load_from_disk(os.path.join(base_dir, 'stage_3'))
    stage_4 = load_from_disk(os.path.join(base_dir, 'stage_4'))
    stage_5_replay = load_from_disk(os.path.join(base_dir, 'stage_5_replay'))
    eval_set = load_from_disk(os.path.join(base_dir, 'eval_set'))

    print(f"\u2705 Success!")
    print(f"Stage 1: {len(stage_1)} samples")
    print(f"Stage 2: {len(stage_2)} samples")
    print(f"Stage 3: {len(stage_3)} samples")
    print(f"Stage 4: {len(stage_4)} samples")
    print(f"Stage 5 (Replay): {len(stage_5_replay)} samples")
except Exception as e:
    print(f"\u274c Error loading datasets: {e}")
    print("Make sure Google Drive is mounted and the paths are correct.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading curriculum stages from /content/drive/MyDrive/math_sft_datasets...
✅ Success!
Stage 1: 20039 samples
Stage 2: 20079 samples
Stage 3: 19434 samples
Stage 4: 37117 samples
Stage 5 (Replay): 17727 samples


In [ ]:
import os

# Check if venv exists, if not, create it
if not os.path.exists('/content/.sft_venv'):
    print("Creating .sft_venv as it was not found...")
    !uv venv .sft_venv --seed
else:
    print("✅ .sft_venv found.")

# Now retry the installation with the correct index for CUDA wheels
requirements_file = '/content/requirements_sft_final.txt'
sft_pip = '/content/.sft_venv/bin/pip'

if os.path.exists(requirements_file):
    print(f"Retrying installation from {requirements_file} with CUDA index...")
    # We add the extra-index-url to find the +cu130 builds
    !{sft_pip} install -r {requirements_file} --extra-index-url https://download.pytorch.org/whl/cu121
else:
    print(f"⚠️ {requirements_file} still missing. Please upload it.")

### Unsloth Integration
We are now installing Unsloth to optimize the fine-tuning process. This library provides specialized kernels that accelerate training and reduce VRAM usage on NVIDIA GPUs like the L4.

In [ ]:
# !/content/.sft_venv/bin/pip install --no-deps "unsloth[cu130-torch2100] @ git+https://github.com/unslothai/unsloth.git"

In [ ]:
# functions for testing the model (validation) after eachstage

import re
import torch
from unsloth import FastLanguageModel

def extract_boxed(text):
    matches = re.findall(r'\\boxed\{((?:[^{}]|(?:\{[^{}]*\}))*)\}', text)
    return matches[-1].strip() if matches else ""

def normalize_answer(ans):
    # strip whitespace, lowercase for letter answers
    return ans.strip().lower().replace(" ", "")

def eval_model(model, tokenizer, eval_set, max_new_tokens=1024):
    FastLanguageModel.for_inference(model)
    model.eval()

    results = {}

    for difficulty, problems in eval_set.items():
        correct = 0
        total = len(problems)

        for problem in problems:
            prompt = tokenizer.apply_chat_template([
                {"role": "system", "content": "You are an expert mathematician. Solve the problem step-by-step."},
                {"role": "user", "content": problem["problem"]},
            ], tokenize=False, add_generation_prompt=True)

            inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    temperature=0.0,      # greedy — deterministic
                    do_sample=False,
                    pad_token_id=tokenizer.eos_token_id,
                )

            generated = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
            pred_answer = extract_boxed(generated)
            true_answer = extract_boxed(problem["solution"])

            if normalize_answer(pred_answer) == normalize_answer(true_answer):
                correct += 1

        acc = correct / total
        results[difficulty] = {"correct": correct, "total": total, "accuracy": acc}
        print(f"  {difficulty:<8}: {correct}/{total} = {acc:.1%}")

    overall = sum(r["correct"] for r in results.values()) / sum(r["total"] for r in results.values())
    results["overall"] = overall
    print(f"  {'overall':<8}: {overall:.1%}")

    FastLanguageModel.for_training(model)
    torch.cuda.empty_cache()

    return results

In [ ]:
from unsloth import FastLanguageModel
import torch
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import concatenate_datasets
from unsloth import is_bfloat16_supported

max_seq_length = 4096

print("1. Loading Model with Unsloth...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen3-4B-Thinking-2507",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
    trust_remote_code = True,
)

print("2. Configuring QLoRA Adapters (r=64, alpha=128)...")
model = FastLanguageModel.get_peft_model(
    model,
    r = 64,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 128,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

print("3. Preparing Stage 1-2 Data...")
# Combine the previously defined stage_1 and stage_2 datasets
stage_1_2_data = concatenate_datasets([stage_1, stage_2]).shuffle(seed=42)

def format_stage_1_2(example):
    text = tokenizer.apply_chat_template([
        {"role": "system", "content": "You are an expert mathematician. Solve the problem step-by-step."},
        {"role": "user", "content": example["problem"]},
        {"role": "assistant", "content": f"<think>\n{example['solution']}\n</think>\n\\boxed{{{example['answer']}}}"},
    ], tokenize=False, add_generation_prompt=False)
    return {"text": text}

stage_1_2_formatted = stage_1_2_data.map(format_stage_1_2)
print(stage_1_2_formatted[0]["text"])

Start of Stage 1-2 Training

In [ ]:


print("4. Starting Stage 1-2 Training...")
trainer_1_2 = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = stage_1_2_formatted,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    args = TrainingArguments(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 16,
        learning_rate = 5e-5,
        max_steps = 300, # Adjust for full epoch based on your total steps
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        output_dir = "outputs_stage1_2",
        seed = 3407,
    ),
)

from unsloth.chat_templates import train_on_responses_only

trainer_1_2 = train_on_responses_only(
    trainer_1_2,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)

trainer_1_2.train()
print("✅ Stage 1 & 2 training complete.")

4. Starting Stage 1-2 Training...


Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/40118 [00:00<?, ? examples/s]

Map (num_proc=16):   0%|          | 0/40118 [00:00<?, ? examples/s]

Filter (num_proc=16):   0%|          | 0/40118 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Unsloth: Removed 3 out of 40118 samples from train_dataset where all labels were -100 (no response found after truncation). This prevents NaN loss during training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 40,115 | Num Epochs = 1 | Total steps = 300
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 16
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 16 x 1) = 16
 "-____-"     Trainable parameters = 132,120,576 of 4,154,588,672 (3.18% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
10,0.544613
20,0.455709
30,0.404431
40,0.449069
50,0.416424
60,0.427610
70,0.434577
80,0.401237
90,0.421034
100,0.406653


Unsloth: Restored added_tokens_decoder metadata in outputs_stage1_2/checkpoint-300/tokenizer_config.json.


✅ Stage 1 & 2 training complete.


In [ ]:
# After trainer_1_2.train()
print("\n=== Eval after Stage 1-2 ===")
results_log["stage_1_2"] = eval_model(model, tokenizer, eval_set)

with open(f"{base_dir}/results_log.json", "w") as f:
    json.dump(results_log, f, indent=2)

Both `max_new_tokens` (=1024) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



=== Eval after Stage 1-2 ===


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API i

KeyboardInterrupt: 

In [ ]:
import gc
import torch

# 1. Save Stage 1-2 LoRA Adapter to Drive
print("Saving Stage 1-2 LoRA to Drive...")
model.save_pretrained("/content/drive/MyDrive/checkpoints/lora_stage1_2")

# 2. Merge weights and save full 16-bit model to Drive
print("Merging weights and saving full model to Drive...")
model.save_pretrained_merged("/content/drive/MyDrive/checkpoints/merged_stage1_2", tokenizer, save_method="merged_16bit")

# 3. Run Evaluation for Stage 1-2
if 'eval_set' in globals() and 'quick_eval' in globals():
    print("Running Stage 1-2 Evaluation...")
    quick_eval(model, tokenizer, eval_set, "stage_1_2")
else:
    print("⚠️ Eval set or quick_eval function not found. Skipping evaluation.")

# 4. VRAM Cleanup: Clear the previous model and trainer to free up the L4 GPU
del model
del trainer_1_2
gc.collect()
torch.cuda.empty_cache()

# 5. Reload the newly merged model as the fresh base for Stage 3
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "/content/drive/MyDrive/checkpoints/merged_stage1_2",
    max_seq_length = 4096,
    load_in_4bit = True,
    trust_remote_code = True
)

print("✅ Stage 1-2 checkpointing complete. Model reloaded for next stage.")

Saving Stage 1-2 LoRA to Drive...
Merging weights and saving full model to Drive...


config.json: 0.00B [00:00, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/checkpoints/merged_stage1_2/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

...


Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [00:54<00:00, 27.41s/it]


Unsloth: Merge process complete. Saved to `/content/drive/MyDrive/checkpoints/merged_stage1_2`
⚠️ Eval set or quick_eval function not found. Skipping evaluation.
Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2026.4.8: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

The tokenizer you are loading from '/content/drive/MyDrive/checkpoints/merged_stage1_2' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
The tokenizer you are loading from '/content/drive/MyDrive/checkpoints/merged_stage1_2' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


✅ Stage 1-2 checkpointing complete. Model reloaded for next stage.


In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig, AutoTokenizer
from peft import get_peft_model, LoraConfig, prepare_model_for_kbit_training
import torch

# 1. Define BitsAndBytesConfig for 4-bit loading
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="fp4",
    bnb_4bit_use_double_quant=True
)

# 2. Load the base model with quantization
model_id = "Qwen/Qwen3-4B-Thinking-2507"
print(f"Loading model {model_id} with 4-bit quantization...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,

)

# 3. Prepare model for k-bit training
model = prepare_model_for_kbit_training(model)

# 4. Define LoRA Configuration
# Targeting attention and MLP layers for comprehensive reasoning fine-tuning
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# 5. Wrap model with LoRA
model = get_peft_model(model, lora_config)

# 6. Verify trainable parameters
model.print_trainable_parameters()
print("\u2705 Model successfully configured with QLoRA.")

Loading model Qwen/Qwen3-4B-Thinking-2507 with 4-bit quantization...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

trainable params: 33,030,144 || all params: 4,055,498,240 || trainable%: 0.8145
✅ Model successfully configured with QLoRA.


In [ ]:
from unsloth import FastLanguageModel
import torch

model_id = "Qwen/Qwen3-4B-Thinking-2507"
max_seq_length = 4096 # As per your plan
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage.

# 1. Load model via Unsloth
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_id,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    trust_remote_code = True,
)

# 2. Add LoRA Adapters with your specific plan parameters
model = FastLanguageModel.get_peft_model(
    model,
    r = 64, # Higher rank as requested
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 128,
    lora_dropout = 0.05, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

print("✅ Model loaded and LoRA configured via Unsloth.")

In [ ]:
from transformers import AutoTokenizer

# 1. Load the tokenizer for the thinking model
model_id = "Qwen/Qwen3-4B-Thinking-2507"
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

# 2. Configure padding for stable Causal LM fine-tuning
# Using 'right' padding is the standard for most decoder-only architectures during SFT
tokenizer.padding_side = "right"

# 3. Handle pad token
# If pad_token is not set, using eos_token is a common and stable practice
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 4. Verification
print(f"Tokenizer loaded from: {model_id}")
print(f"Padding side: {tokenizer.padding_side}")
print(f"Pad token: {tokenizer.pad_token} (ID: {tokenizer.pad_token_id})")
print(f"EOS token: {tokenizer.eos_token} (ID: {tokenizer.eos_token_id})")

Tokenizer loaded from: Qwen/Qwen3-4B-Thinking-2507
Padding side: right
Pad token: <|endoftext|> (ID: 151643)
EOS token: <|im_end|> (ID: 151645)


In [ ]:
def format_math_dataset(example):
    # 1. Define the system prompt consistent with the model's expert persona
    system_prompt = "You are an expert mathematician. Solve the problem step-by-step."

    # 2. Construct messages list with reasoning tags for the assistant
    # We wrap the solution in <thought> and <solution> tags as per Qwen-Thinking expectations
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": example["problem"]},
        {"role": "assistant", "content": f"<thought>\n{example['solution']}\n</thought>\n<solution>\n\\boxed{{{example['answer']}}}\n</solution>"}
    ]

    # 3. Apply the chat template using the previously initialized tokenizer
    # We set tokenize=False because the SFTTrainer typically handles tokenization
    formatted_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    return {"text": formatted_text}

# 4. Map the function across the subsampled dataset
print("Formatting dataset for SFT...")
formatted_sft_dataset = sft_train_subset.map(
    format_math_dataset,
    remove_columns=sft_train_subset.column_names
)

# 5. Verify the format
print("\u2705 Dataset formatted successfully.")
print("\n--- Formatted Sample Preview ---")
print(formatted_sft_dataset[0]['text'][:1000] + "...")

Formatting dataset for SFT...
✅ Dataset formatted successfully.

--- Formatted Sample Preview ---
<|im_start|>system
You are an expert mathematician. Solve the problem step-by-step.<|im_end|>
<|im_start|>user
Define the sequence $(a_n)$ as $a_1 = 1$, $a_{2n} = a_n$ and $a_{2n+1} = a_n + 1$ for all $n\geq 1$.

a) Find all positive integers $n$ such that $a_{kn} = a_n$ for all integers $1 \leq k \leq n$.
b) Prove that there exist infinitely many positive integers $m$ such that $a_{km} \geq a_m$ for all positive integers $k$.<|im_end|>
<|im_start|>assistant
<think>

</think>

<thought>
First, we need to understand the sequence $(a_n)$. Notice the following lemma:

**Lemma**: \(a_n = s_2(n)\), where \(s_2\) is the sum of the digits of \(n\) written in binary.

This lemma can be proved by induction on \(n\). For the base case, \(a_1 = 1\) and \(s_2(1) = 1\). For the inductive step, assume \(a_n = s_2(n)\) holds for some \(n\). Then:
- \(a_{2n} = a_n = s_2(n) = s_2(2n)\) because multiplying 

In [ ]:
import sys
import os

# Ensure the SFT venv site-packages are at the front of sys.path
sft_site_packages = '/content/.sft_venv/lib/python3.12/site-packages'
if sft_site_packages not in sys.path:
    sys.path.insert(0, sft_site_packages)

from trl import SFTTrainer, SFTConfig
import torch

# 1. Define SFTConfig with memory optimizations and dataset configs
training_args = SFTConfig(
    output_dir="./qwen-sft-results",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    max_steps=500,
    logging_steps=10,
    optim="paged_adamw_8bit",
    bf16=True,
    gradient_checkpointing=True,
    save_strategy="steps",
    save_steps=50,
    report_to="none",
    dataset_text_field="text",
    max_seq_length=8192  # Reduced slightly to ensure no padding issues
)

# 2. Initialize the SFTTrainer
print("Initializing SFTTrainer...")
trainer = SFTTrainer(
    model=model,
    train_dataset=formatted_sft_dataset,
    tokenizer=tokenizer,
    args=training_args
)

# 3. Start the fine-tuning process
print("Starting training loop...")
trainer.train()

print("\u2705 Fine-tuning complete!")

Initializing SFTTrainer...
Starting training loop...


Step,Training Loss
10,0.894030
20,0.859674
30,0.751635
40,0.771257
50,0.691139
60,0.776422
70,0.773632
80,0.688853
90,0.711307
100,0.694930


✅ Fine-tuning complete!


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

# Updated Trainer with Unsloth optimizations
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = formatted_sft_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Reasoning traces have meaningful boundaries
    args = TrainingArguments(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 16,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

# Start training test
trainer_stats = trainer.train()
print("✅ Unsloth test training complete!")

## Curriculum Learning: Multi-Stage SFT

This section implements the multi-stage training curriculum. We utilize Unsloth for performance and manually merge weights between major stages to refresh LoRA capacity.

### Stage 1 & 2: Base LoRA (Easy to Medium)

In [ ]:
from datasets import load_dataset, concatenate_datasets
import re
import torch
import gc
from unsloth import FastLanguageModel
from tqdm import tqdm

# 1. Updated Verification Helper
def is_valid(example):
    thinking = example.get('solution', '')
    boxed_match = r'\\boxed{' in example.get('solution', '') or r'\\boxed{' in example.get('answer', '')
    cot_word_count = len(thinking.split())
    return 150 < cot_word_count < 2000 and boxed_match

# 2. Master Dataset Loader
print("Loading curriculum datasets...")
open_r1 = load_dataset("open-r1/OpenR1-Math-220k", "default", split="train")

# 3. Tiering by complexity
filtered_data = open_r1.filter(is_valid)
sorted_data = filtered_data.map(lambda x: {'sol_len': len(x['solution'])}).sort('sol_len')

total = len(sorted_data)
stage_1_2_data = sorted_data.select(range(0, int(total * 0.4)))  # Bottom 40% (Easy/Medium)
stage_3_4_data = sorted_data.select(range(int(total * 0.4), int(total * 0.9)))  # Top 40-90% (Hard)

# 4. Create Held-out Eval Set
eval_set = sorted_data.select(range(int(total * 0.9), int(total * 0.9) + 100)) # Top 100 from highest difficulty for eval

print(f"Dataset Tiers Prepared:\n- Stage 1-2 (Simpler): {len(stage_1_2_data)}\n- Stage 3-4 (Complex): {len(stage_3_4_data)}\n- Eval Set: {len(eval_set)}")

# 5. Quick Eval Function
def quick_eval(model, tokenizer, eval_set, stage_name):
    gc.collect()
    torch.cuda.empty_cache()

    FastLanguageModel.for_inference(model)
    correct = {"easy": 0, "medium": 0, "hard": 0}
    total_counts = {"easy": 0, "medium": 0, "hard": 0}

    for item in tqdm(eval_set, desc=f"Eval {stage_name}"):
        slen = item['sol_len']
        if slen < 500: diff = "easy"
        elif slen < 1500: diff = "medium"
        else: diff = "hard"
        total_counts[diff] += 1

        prompt_text = tokenizer.apply_chat_template([
            {"role": "system", "content": "You are an expert mathematician. Solve the problem step-by-step."},
            {"role": "user", "content": item["problem"]}
        ], tokenize=False, add_generation_prompt=True)

        inputs = tokenizer(prompt_text, return_tensors="pt").to("cuda")
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=1024, temperature=0, pad_token_id=tokenizer.pad_token_id)

        resp = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
        match = re.search(r'\\boxed\{((?:[^{}]|(?:\{[^{}]*\}))*)\}', resp)
        if not match: match = re.search(r'\\boxed\{(.*?)\}', resp)
        pred = match.group(1).strip() if match else ""

        gold = str(item['answer']).strip()
        if pred == gold:
            correct[diff] += 1

    FastLanguageModel.for_training(model)
    gc.collect()
    torch.cuda.empty_cache()

    acc = {k: (correct[k] / total_counts[k] if total_counts[k] > 0 else 0.0) for k in correct}
    print(f"\n--- Eval Results for {stage_name} ---")
    for k in ["easy", "medium", "hard"]:
        print(f"{k.capitalize()}: {acc[k]:.2%} ({correct[k]}/{total_counts[k]})")

    results_log[stage_name] = acc
    with open('/content/drive/MyDrive/checkpoints/results_log.json', 'w') as f:
        json.dump(results_log, f, indent=4)

    return acc

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported

# 1. Format Stage 1-2 Data with correct <think> tags
def format_prompt(example):
    return {"text": tokenizer.apply_chat_template([
        {"role": "system", "content": "You are an expert mathematician. Solve the problem step-by-step."},
        {"role": "user", "content": example["problem"]},
        {"role": "assistant", "content": f"<think>\n{example['solution']}\n</think>\n\\boxed{{{example['answer']}}}"}
    ], tokenize=False)}

stage_1_2_formatted = stage_1_2_data.map(format_prompt)

# 2. Stage 1-2 Training Execution with lowered learning rate
trainer_1_2 = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = stage_1_2_formatted,
    dataset_text_field = "text",
    max_seq_length = 4096,
    args = SFTConfig(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 16,
        learning_rate = 1e-5,
        max_steps = 200,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        output_dir = "outputs_stage1_2",
    ),
)

trainer_1_2.train()
print("✅ Stage 1 & 2 complete. Correct tags used and LR set to 1e-5.")

In [ ]:
# Quick gen test - Sanity Check for Reasoning Tags
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)

# Test a simple problem to verify <think> and \boxed{} behavior
inputs = tokenizer.apply_chat_template([
    {"role": "system", "content": "You are an expert mathematician. Solve the problem step-by-step."},
    {"role": "user", "content": "Solve for x: 2x + 3 = 11"}
], return_tensors="pt", add_generation_prompt=True).to("cuda")

outputs = model.generate(
    input_ids=inputs.input_ids,
    attention_mask=inputs.attention_mask,
    max_new_tokens=512,
    temperature=0.1
)

print("--- Sanity Check Output ---")
print(tokenizer.decode(outputs[0], skip_special_tokens=False))

# Return to training mode
FastLanguageModel.for_training(model)

### Merge & Refresh LoRA (Stage 2 -> Stage 3)

In [ ]:
import gc

# 1. Save LoRA Adapter to Drive
print("Saving Stage 1-2 LoRA to Drive...")
model.save_pretrained("/content/drive/MyDrive/checkpoints/lora_stage1_2")

# 2. Merge weights and save to Drive
print("Merging LoRA weights from Stage 1-2 and saving to Drive...")
model.save_pretrained_merged("/content/drive/MyDrive/checkpoints/merged_stage1_2", tokenizer, save_method = "merged_16bit")

# 3. Eval Stage 1-2
print("Evaluating Stage 1-2...")
quick_eval(model, tokenizer, eval_set, "stage_1_2")

# 4. VRAM Cleanup
del model
del trainer_1_2
gc.collect()
torch.cuda.empty_cache()

# 5. Reload Merged Model as Fresh Base
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "/content/drive/MyDrive/checkpoints/merged_stage1_2",
    max_seq_length = 4096,
    load_in_4bit = True,
)

print("✅ Base model refreshed with Stage 1-2 knowledge. VRAM cleared.")

In [1]:
import os
import sys
import random
import json
import re
from pathlib import Path

import torch
from tqdm import tqdm

# Local Lightning workspace paths
COMP_DIR = Path("comp_folder/151B_SP26_Competition")
DATA_PATH = COMP_DIR / "data/public.jsonl"
JUDGER_DIR = COMP_DIR
VANILLA_RESULTS_PATH = COMP_DIR / "results/12k_thinking/starter_results_12k.jsonl"
OUTPUT_DIR = COMP_DIR / "results/stage1_2_eval"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BASE_MODEL = "Qwen/Qwen3-4B-Thinking-2507"
LORA_PATH = Path("models/lora_stage1_2")
SAMPLE_SIZE = 50
SEED = 42
MAX_NEW_TOKENS = 2048

# Make competition judger importable from the local folder.
if str(JUDGER_DIR.resolve()) not in sys.path:
    sys.path.insert(0, str(JUDGER_DIR.resolve()))

from judger import Judger
from unsloth import FastLanguageModel

# Reuse the model/tokenizer if you already ran the top LoRA-loading cell.
# Otherwise, load base Qwen + the local Stage 1-2 LoRA adapter.
if "model" not in globals() or "tokenizer" not in globals():
    print("Loading base model + Stage 1-2 LoRA adapter...")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL,
        max_seq_length=4096,
        dtype=None,
        load_in_4bit=True,
        trust_remote_code=True,
    )
    model.load_adapter(str(LORA_PATH))
else:
    print("Using existing model/tokenizer from the notebook session.")

FastLanguageModel.for_inference(model)
judger = Judger(strict_extract=False)

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Missing public data: {DATA_PATH}")
if not VANILLA_RESULTS_PATH.exists():
    raise FileNotFoundError(f"Missing vanilla baseline results: {VANILLA_RESULTS_PATH}")

with DATA_PATH.open() as f:
    original_data = [json.loads(line) for line in f]

with VANILLA_RESULTS_PATH.open() as f:
    vanilla_results = {r["id"]: r for r in (json.loads(line) for line in f)}

rng = random.Random(SEED)
sample_data = rng.sample(original_data, min(SAMPLE_SIZE, len(original_data)))

SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. Solve the problem step-by-step. "
    "Put your final answer inside \\boxed{}."
)
SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician. Read the problem and answer choices. "
    "Output only the chosen option letter inside \\boxed{}, e.g. \\boxed{C}."
)

def build_user_prompt(item):
    if item.get("options"):
        labels = [chr(65 + i) for i in range(len(item["options"]))]
        opts_text = "\n".join(f"{label}. {opt.strip()}" for label, opt in zip(labels, item["options"]))
        return SYSTEM_PROMPT_MCQ, f"{item['question']}\n\nOptions:\n{opts_text}"
    return SYSTEM_PROMPT_MATH, item["question"]

def extract_boxed(text):
    matches = re.findall(r"\\boxed\{((?:[^{}]|(?:\{[^{}]*\}))*)\}", text)
    if matches:
        return matches[-1].strip()
    simple = re.findall(r"\\boxed\{(.*?)\}", text)
    return simple[-1].strip() if simple else ""

def extract_mcq_letter(text):
    boxed = extract_boxed(text).strip().upper()
    if re.fullmatch(r"[A-Z]", boxed):
        return boxed
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""

def score_response(item, response):
    gold = item["answer"]
    if item.get("options"):
        return extract_mcq_letter(response) == str(gold).strip().upper()

    gold_list = gold if isinstance(gold, list) else [gold]
    try:
        return bool(judger.auto_judge(pred=response, gold=gold_list, options=[[]] * len(gold_list)))
    except Exception as exc:
        print(f"Judger failed for id={item.get('id')}: {exc}")
        return False

stage_correct = 0
vanilla_correct = 0
comparison_logs = []

print(f"Running Stage 1-2 inference on {len(sample_data)} sampled questions...")
for item in tqdm(sample_data):
    q_id = item.get("id")
    system_prompt, user_prompt = build_user_prompt(item)
    prompt_text = tokenizer.apply_chat_template(
        [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(prompt_text, return_tensors="pt").to("cuda" if torch.cuda.is_available() else "cpu")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
        )

    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
    correct = score_response(item, response)
    vanilla_res = vanilla_results.get(q_id, {})
    vanilla_is_correct = bool(vanilla_res.get("correct", False))

    stage_correct += int(correct)
    vanilla_correct += int(vanilla_is_correct)

    comparison_logs.append({
        "id": q_id,
        "is_mcq": bool(item.get("options")),
        "question": item.get("question"),
        "options": item.get("options"),
        "gold": item.get("answer"),
        "stage1_2_response": response,
        "stage1_2_correct": correct,
        "vanilla_response": vanilla_res.get("response"),
        "vanilla_correct": vanilla_is_correct,
    })

sample_n = len(sample_data)
stage_acc = stage_correct / sample_n if sample_n else 0.0
vanilla_acc = vanilla_correct / sample_n if sample_n else 0.0

results_path = OUTPUT_DIR / "stage1_2_vs_vanilla_50_sample.jsonl"
summary_path = OUTPUT_DIR / "stage1_2_vs_vanilla_50_sample_summary.json"

with results_path.open("w") as f:
    for row in comparison_logs:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

summary = {
    "sample_size": sample_n,
    "seed": SEED,
    "data_path": str(DATA_PATH),
    "vanilla_results_path": str(VANILLA_RESULTS_PATH),
    "lora_path": str(LORA_PATH),
    "vanilla_correct": vanilla_correct,
    "vanilla_accuracy": vanilla_acc,
    "stage1_2_correct": stage_correct,
    "stage1_2_accuracy": stage_acc,
    "delta_accuracy": stage_acc - vanilla_acc,
    "results_path": str(results_path),
}
with summary_path.open("w") as f:
    json.dump(summary, f, indent=2)

print("\n" + "=" * 60)
print(f"ACCURACY COMPARISON (sample size: {sample_n})")
print(f"Vanilla:   {vanilla_correct}/{sample_n} ({vanilla_acc:.1%})")
print(f"Stage 1-2: {stage_correct}/{sample_n} ({stage_acc:.1%})")
print(f"Delta:     {stage_acc - vanilla_acc:+.1%}")
print(f"Saved per-question results to: {results_path}")
print(f"Saved summary to: {summary_path}")
print("=" * 60)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
Loading base model + Stage 1-2 LoRA adapter...
Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2026.4.8: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

unsloth/qwen3-4b-thinking-2507-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Loading weights:   0%|          | 0/504 [00:00<?, ?it/s]

Running Stage 1-2 inference on 50 sampled questions...


  0%|          | 0/50 [00:00<?, ?it/s]Both `max_new_tokens` (=2048) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers


ACCURACY COMPARISON (sample size: 50)
Vanilla:   28/50 (56.0%)
Stage 1-2: 13/50 (26.0%)
Delta:     -30.0%
Saved per-question results to: comp_folder/151B_SP26_Competition/results/stage1_2_eval/stage1_2_vs_vanilla_50_sample.jsonl
Saved summary to: comp_folder/151B_SP26_Competition/results/stage1_2_eval/stage1_2_vs_vanilla_50_sample_summary.json


In [1]:
import re
import json

def count_ans_placeholders(question_text):
    return question_text.count("[ANS]")

def extract_all_boxed(text):
    """Extract all \boxed{} contents in order of appearance."""
    pattern = r'\\boxed\{((?:[^{}]|(?:\{[^{}]*\}))*)\}'
    return re.findall(pattern, text)

def extract_think_block(text):
    """Get content inside <think>...</think>."""
    match = re.search(r'<think>(.*?)</think>', text, re.DOTALL)
    return match.group(1) if match else ""

def extract_after_think(text):
    """Get content after </think>."""
    parts = text.split('</think>')
    return parts[-1].strip() if len(parts) > 1 else text

def postprocess_output(generated, question_text, is_mcq=False, options=None):
    """
    Fix LoRA output for multi-answer and MCQ questions.
    Returns corrected final answer string.
    """
    n_expected = count_ans_placeholders(question_text)
    
    # Extract think block and post-think separately
    think_content = extract_think_block(generated)
    after_think = extract_after_think(generated)
    
    # All boxes from inside think
    think_boxes = extract_all_boxed(think_content)
    # Boxes from after think
    final_boxes = extract_all_boxed(after_think)
    
    # --- MCQ handling ---
    if is_mcq and options:
        valid_letters = [chr(65 + i) for i in range(len(options))]
        
        # Check final boxes first
        for box in reversed(final_boxes + think_boxes):
            cleaned = box.strip().upper()
            if cleaned in valid_letters:
                return f"\\boxed{{{cleaned}}}"
        
        # Fallback: scan full text for last valid letter mention
        all_letters = re.findall(r'\b([A-J])\b', generated.upper())
        valid_mentions = [l for l in all_letters if l in valid_letters]
        if valid_mentions:
            return f"\\boxed{{{valid_mentions[-1]}}}"
        
        return after_think.strip() if after_think else generated

    # --- Multi-answer handling ---
    if n_expected > 1:
        all_boxes = think_boxes  # use think boxes — more complete
        
        if len(all_boxes) >= n_expected:
            # Take last n_expected boxes (most likely the final answers per part)
            selected = all_boxes[-n_expected:]
            # Clean each: strip LaTeX text wrappers
            cleaned = []
            for b in selected:
                b = re.sub(r'\\text\{([^}]*)\}', r'\1', b)
                b = b.strip()
                cleaned.append(b)
            return f"\\boxed{{{', '.join(cleaned)}}}"
        
        elif len(all_boxes) > 0:
            # Fewer boxes than expected — join what we have
            cleaned = []
            for b in all_boxes:
                b = re.sub(r'\\text\{([^}]*)\}', r'\1', b)
                b = b.strip()
                cleaned.append(b)
            return f"\\boxed{{{', '.join(cleaned)}}}"
    
    # --- Single answer ---
    if final_boxes:
        return f"\\boxed{{{final_boxes[-1]}}}"
    if think_boxes:
        return f"\\boxed{{{think_boxes[-1]}}}"
    
    return after_think.strip() if after_think else generated


def postprocess_results(input_path, output_path, data_path):
    """
    Apply postprocessing to all LoRA results.
    input_path: jsonl with LoRA responses
    data_path: original public.jsonl with questions
    """
    original = {
        item['id']: item 
        for item in [json.loads(l) for l in open(data_path)]
    }
    
    results = [json.loads(l) for l in open(input_path)]
    fixed = []
    
    improved = 0
    total = 0
    
    for r in results:
        qid = r['id']
        item = original.get(qid, {})
        question = item.get('question', '')
        options = item.get('options', None)
        is_mcq = bool(options)
        
        original_response = r.get('response', '')
        fixed_answer = postprocess_output(
            original_response, 
            question,
            is_mcq=is_mcq,
            options=options
        )
        
        r['response_original'] = original_response
        r['response_fixed'] = fixed_answer
        fixed.append(r)
        total += 1
    
    with open(output_path, 'w') as f:
        for r in fixed:
            f.write(json.dumps(r) + '\n')
    
    print(f"Postprocessed {total} results → {output_path}")
    return fixed


# Usage
fixed_results = postprocess_results(
    input_path="comp_folder/151B_SP26_Competition/results/stage1_2_eval/stage1_2_vs_vanilla_50_sample.jsonl",
    output_path="comp_folder/151B_SP26_Competition/results/stage1_2_eval/lora_fixed.jsonl",
    data_path="/teamspace/studios/this_studio/comp_folder/151B_SP26_Competition/data/public.jsonl"
)

Postprocessed 50 results → comp_folder/151B_SP26_Competition/results/stage1_2_eval/lora_fixed.jsonl


In [10]:
import sys
import json
import re
from pathlib import Path

from tqdm import tqdm

COMP_DIR = Path("comp_folder/151B_SP26_Competition")
DATA_PATH = COMP_DIR / "data/public.jsonl"
JUDGER_DIR = COMP_DIR

FIXED_PATH = Path("/teamspace/studios/this_studio/comp_folder/151B_SP26_Competition/results/stage1_2_eval/lora_fixed.jsonl")
OUT_PATH = Path("/teamspace/studios/this_studio/comp_folder/151B_SP26_Competition/results/stage1_2_eval/lora_fixed_scored.jsonl")
SUMMARY_PATH = Path("/teamspace/studios/this_studio/comp_folder/151B_SP26_Competition/results/stage1_2_eval/lora_fixed_summary.jsonl")

if str(JUDGER_DIR.resolve()) not in sys.path:
    sys.path.insert(0, str(JUDGER_DIR.resolve()))

from judger import Judger

judger = Judger(strict_extract=False)

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Missing public data: {DATA_PATH}")
if not FIXED_PATH.exists():
    raise FileNotFoundError(f"Missing fixed LoRA file: {FIXED_PATH}")

with DATA_PATH.open() as f:
    gold_by_id = {row["id"]: row for row in (json.loads(line) for line in f)}

with FIXED_PATH.open() as f:
    fixed_rows = [json.loads(line) for line in f]


def extract_boxed(text):
    matches = re.findall(r"\\boxed\{((?:[^{}]|(?:\{[^{}]*\}))*)\}", text or "")
    if matches:
        return matches[-1].strip()

    simple = re.findall(r"\\boxed\{(.*?)\}", text or "")
    return simple[-1].strip() if simple else ""


def extract_mcq_letter(text):
    boxed = extract_boxed(text).strip().upper()
    if re.fullmatch(r"[A-Z]", boxed):
        return boxed

    # Fallback for responses like "Answer: C"
    matches = re.findall(r"\b([A-Z])\b", (text or "").upper())
    return matches[-1] if matches else ""


def get_response(row):
    # Prefer your postprocessed/fixed response if you used a new field name.
    for key in [
        "fixed_response",
        "postprocessed_response",
        "stage1_2_fixed_response",
        "stage1_2_response",
        "response",
    ]:
        if key in row and row[key] is not None:
            return row[key]
    return ""


def score_response(item, response):
    gold = item["answer"]

    # MCQ public data stores options; gold is a letter.
    if item.get("options"):
        return extract_mcq_letter(response) == str(gold).strip().upper()

    gold_list = gold if isinstance(gold, list) else [gold]

    try:
        return bool(
            judger.auto_judge(
                pred=response,
                gold=gold_list,
                options=[[]] * len(gold_list),
            )
        )
    except Exception as exc:
        print(f"Judger failed for id={item.get('id')}: {exc}")
        return False


scored_rows = []
correct = 0
mcq_total = mcq_correct = 0
free_total = free_correct = 0

for row in tqdm(fixed_rows, desc="Scoring lora_fixed.jsonl"):
    q_id = row["id"]

    if q_id not in gold_by_id:
        raise KeyError(f"id={q_id} not found in public.jsonl")

    item = gold_by_id[q_id]
    response = get_response(row)
    is_mcq = bool(item.get("options"))

    is_correct = score_response(item, response)

    correct += int(is_correct)

    if is_mcq:
        mcq_total += 1
        mcq_correct += int(is_correct)
    else:
        free_total += 1
        free_correct += int(is_correct)

    scored = dict(row)
    scored["gold"] = item["answer"]
    scored["is_mcq"] = is_mcq
    scored["fixed_response"] = response
    scored["fixed_correct"] = is_correct

    scored_rows.append(scored)

n = len(scored_rows)
acc = correct / n if n else 0.0
mcq_acc = mcq_correct / mcq_total if mcq_total else 0.0
free_acc = free_correct / free_total if free_total else 0.0

with OUT_PATH.open("w") as f:
    for row in scored_rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

summary = {
    "input_path": str(FIXED_PATH),
    "output_path": str(OUT_PATH),
    "sample_size": n,
    "correct": correct,
    "accuracy": acc,
    "mcq_total": mcq_total,
    "mcq_correct": mcq_correct,
    "mcq_accuracy": mcq_acc,
    "free_total": free_total,
    "free_correct": free_correct,
    "free_accuracy": free_acc,
}

with SUMMARY_PATH.open("w") as f:
    json.dump(summary, f, indent=2)

print("\n" + "=" * 60)
print("LORA FIXED EVAL")
print(f"Overall: {correct}/{n} ({acc:.1%})")
print(f"MCQ:     {mcq_correct}/{mcq_total} ({mcq_acc:.1%})")
print(f"Free:    {free_correct}/{free_total} ({free_acc:.1%})")
print(f"Scored rows: {OUT_PATH}")
print(f"Summary:     {SUMMARY_PATH}")
print("=" * 60)

Scoring lora_fixed.jsonl:   0%|          | 0/50 [00:00<?, ?it/s]

Scoring lora_fixed.jsonl: 100%|██████████| 50/50 [00:02<00:00, 20.72it/s]


LORA FIXED EVAL
Overall: 13/50 (26.0%)
MCQ:     6/20 (30.0%)
Free:    7/30 (23.3%)
Scored rows: /teamspace/studios/this_studio/comp_folder/151B_SP26_Competition/results/stage1_2_eval/lora_fixed_scored.jsonl
Summary:     /teamspace/studios/this_studio/comp_folder/151B_SP26_Competition/results/stage1_2_eval/lora_fixed_summary.jsonl


In [ ]:
# Quick gen test
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)

inputs = tokenizer.apply_chat_template([
    {"role": "user", "content": "Solve: 2x + 3 = 11"}
], return_tensors="pt").to("cuda")

outputs = model.generate(input_ids=inputs, max_new_tokens=512)
print(tokenizer.decode(outputs[0], skip_special_tokens=False))

### Stage 3 & 4: Fresh LoRA (Hard / AIME)

In [ ]:
# 1. Initialize FRESH LoRA on the merged model
model = FastLanguageModel.get_peft_model(
    model,
    r = 64,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 128,
    lora_dropout = 0.05,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

# 2. Format Stage 3-4 Data (Harder reasoning samples)
stage_3_4_formatted = stage_3_4_data.map(format_prompt)

# 3. Stage 3-4 Training Execution
trainer_3_4 = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = stage_3_4_formatted,
    dataset_text_field = "text",
    max_seq_length = 4096,
    args = SFTConfig(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 16,
        learning_rate = 1e-5, # Slightly lower learning rate for harder tier
        max_steps = 200,
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        output_dir = "outputs_stage3_4",
    ),
)

trainer_3_4.train()
print("✅ Stage 3 & 4 (Hard Tier) complete!")

In [ ]:
# Quick gen test
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)

inputs = tokenizer.apply_chat_template([
    {"role": "user", "content": "Solve: 2x + 3 = 11"}
], return_tensors="pt").to("cuda")

outputs = model.generate(input_ids=inputs, max_new_tokens=512)
print(tokenizer.decode(outputs[0], skip_special_tokens=False))

In [ ]:
import gc

# 1. Save Stage 3-4 LoRA to Drive
print("Saving Stage 3-4 LoRA to Drive...")
model.save_pretrained("/content/drive/MyDrive/checkpoints/lora_stage3_4")

# 2. Final Merge of Stage 3-4 to Drive
print("Merging Stage 3-4 weights and saving to Drive...")
model.save_pretrained_merged("/content/drive/MyDrive/checkpoints/merged_stage3_4", tokenizer, save_method = "merged_16bit")

# 3. Eval Stage 3-4
print("Evaluating Stage 3-4...")
quick_eval(model, tokenizer, eval_set, "stage_3_4")

# 4. VRAM Cleanup
del model
del trainer_3_4
gc.collect()
torch.cuda.empty_cache()

# 5. Reload for Final Stage
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "/content/drive/MyDrive/checkpoints/merged_stage3_4",
    max_seq_length = 4096,
    load_in_4bit = True,
)

# 6. Final Fresh LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 64,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

# 7. Format Stage 5 (Replay)
stage_5_formatted = stage_5_replay.map(format_prompt)

# 8. Stage 5 Training Execution
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported
trainer_5 = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = stage_5_formatted,
    dataset_text_field = "text",
    max_seq_length = 4096,
    args = SFTConfig(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 16,
        learning_rate = 5e-6,
        max_steps = 100,
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        output_dir = "outputs_stage5_final",
    ),
)

trainer_5.train()
print("✅ Stage 5 Replay complete.")

In [ ]:
# Quick gen test
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)

inputs = tokenizer.apply_chat_template([
    {"role": "user", "content": "Solve: 2x + 3 = 11"}
], return_tensors="pt").to("cuda")

outputs = model.generate(input_ids=inputs, max_new_tokens=512)
print(tokenizer.decode(outputs[0], skip_special_tokens=False))

In [ ]:
import gc
import json
from datasets import concatenate_datasets

print("Preparing Stage 5 Replay Dataset...")
s12_sample = stage_1_2_data.shuffle(seed=42).select(range(min(1000, len(stage_1_2_data))))
s34_sample = stage_3_4_data.shuffle(seed=42).select(range(min(1000, len(stage_3_4_data))))
stage_5_replay_data = concatenate_datasets([s12_sample, s34_sample]).shuffle(seed=42)
stage_5_formatted = stage_5_replay_data.map(format_prompt)

print("Starting Stage 5: Final Stabilization Replay...")
trainer_5 = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = stage_5_formatted,
    dataset_text_field = "text",
    max_seq_length = 4096,
    args = SFTConfig(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 16,
        learning_rate = 5e-6,
        max_steps = 100,
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        output_dir = "outputs_stage5_final",
        save_strategy = "no",
    ),
)
trainer_5.train()

print("Saving Stage 5 LoRA to Drive...")
model.save_pretrained("/content/drive/MyDrive/checkpoints/lora_stage5")

print("Finalizing model: Merging all stages and saving to Drive...")
model.save_pretrained_merged("/content/drive/MyDrive/checkpoints/merged_stage5", tokenizer, save_method = "merged_16bit")

print("Evaluating Stage 5...")
quick_eval(model, tokenizer, eval_set, "stage_5")

print("\n=======================================================")
print("           CURRICULUM ACCURACY PROGRESSION             ")
print("=======================================================")
print(f"{'Stage':<15} | {'Easy':<10} | {'Medium':<10} | {'Hard':<10}")
print("-

In [ ]:
# Quick gen test
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)

inputs = tokenizer.apply_chat_template([
    {"role": "user", "content": "Solve: 2x + 3 = 11"}
], return_tensors="pt").to("cuda")

outputs = model.generate(input_ids=inputs, max_new_tokens=512)
print(tokenizer.decode(outputs[0], skip_special_tokens=False))

### Final Model Inference Test
Use this cell to test your final fine-tuned model on a complex math problem.